# CIFAR-10-C: Conformal Predikció + Drift Detektálás


In [1]:
import os
import math
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

## 1. Adatok betöltése

Két adathalmaz lesz

1. **Clean CIFAR-10**: tanítjuk a modellt, ebből kalibráljuk a detektorokat
2. **Noisy CIFAR-10-C**: Gauss zajos képek, ezeken teszteljük, hogy a detektor detektal-e

A '.npy' fájlokat (zajos adat+label) a './data/" mappában kell elhelyezni (CIFAR-10-C nem töltődik leautomatikusan.)

In [2]:
# Egyszerű normalizálás: pixel értékek [0,1] tartományba
transform = transforms.Compose([
    transforms.ToTensor()
])

# Tiszta CIFAR-10 betöltése (automatikusan letölti, ha még nincs meg)
clean_trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
clean_testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Zajos CIFAR-10-C betöltése
noisy_x_path = os.path.join("./data", "gaussian_noise.npy")
noisy_y_path = os.path.join("./data", "labels.npy")

noisy_X = np.load(noisy_x_path)  # shape: (N, 32, 32, 3)
noisy_y = np.load(noisy_y_path)

# Át kell alakítani PyTorch-os CHW formátumra, és [0,1] skálára hozni
noisy_X_tensor = torch.tensor(noisy_X, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
noisy_y_tensor = torch.tensor(noisy_y, dtype=torch.long)
noisy_dataset  = TensorDataset(noisy_X_tensor, noisy_y_tensor)

c:\Users\anna.gavrilova.TENZORTECH\uni\machine_learning\drift_conf_active_learning\new\dacal-stream-learning\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## 2. cnn-modell: SimpleCNN

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 32×32 → 16×16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16×16 → 8×8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                # 64×8×8 = 4096
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),                             
            nn.Linear(256, num_classes),                
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    def get_embeddings(self, x):
        """256-dimenziós belső reprezentáció visszaadása (az utolsó réteg előtt)."""
        h = self.features(x)
        h = self.classifier[0](h)   # Flatten
        h = self.classifier[1](h)   # Linear: 4096 → 256
        h = self.classifier[2](h)   # ReLU
        return h                    # (N, 256)

## 3. Tanítás és kiértékelés

Szokásos train/eval ciklus.

In [4]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()           
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()                 
        optimizer.step()                
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()  # dropout kikapcsolva értékelésnél
    total_loss, correct = 0, 0
    with torch.no_grad():  # nem kell gradiens (gyorsabb és kevesebb memória)
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

## 4. Drift Detektorok

különböző módszert néztem, mindhárom label nélkül is működik (ez nekünk jó!)


In [5]:
from scipy.stats import ks_2samp  # Kolmogorov-Smirnov teszt


# 1. PCA 
# A modell 256-dimenziós reprezentációkat ad ki. Ez sok, MMD nem működik jól nagy dimenzióban.
#Ezért először csökkentjük 32 dimenzióra PCA-val.

class OnlinePCA:
    """Referencia adatokon tanítja be a PCA-t, majd arra vetíti az új adatokat."""
    def __init__(self, n_components=32):
        self.n_components = n_components
        self.mean_ = None
        self.components_ = None

    def fit(self, X: torch.Tensor):
        X = X.float()
        self.mean_ = X.mean(0)              # átlag kiszámítása
        Xc = X - self.mean_                 # centrálás
        _, _, Vt = torch.linalg.svd(Xc, full_matrices=False)
        self.components_ = Vt[:self.n_components]  # (n_components, d) – a legjobb irányok
        return self

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        # Vetítés a PCA-térbe: centrálás + szorzás az irányokkal
        return (X.float() - self.mean_) @ self.components_.T


# 2. MMD Maximum Mean Discrepancy
# Intuitívan: Ha két adathalmazból véletlenszerűen veszel mintákat és ezek hasonlóak egy statisztika szerint, 
# akkor a halmazok valószínűleg ugyanonnan jönnek.

def _median_sigma(X):
    """Mediánheurisztika: a sigma értéke az összes pontpár-távolság mediánja."""
    if len(X) > 500:
        idx = torch.randperm(len(X))[:500]  # nagy adatnál mintavételezés a sebesség miatt
        X = X[idx]
    dists = torch.cdist(X, X)  # páronkénti euklideszi távolságok
    return dists.median().item() + 1e-6  # kis epsilon a nullaosztás ellen


def _rbf_kernel(A, B, sigma):
    """RBF (Gauss) kernel mátrix: K[i,j] = exp(-||A[i]-B[j]||² / 2σ²)"""
    AA = (A ** 2).sum(1, keepdim=True)
    BB = (B ** 2).sum(1, keepdim=True)
    dist = AA + BB.T - 2 * A @ B.T  # ||a-b||² = ||a||² + ||b||² - 2a·b
    return torch.exp(-dist / (2 * sigma ** 2))


def mmd_statistic(X_ref, X_new, sigma=None):
    """Torzítatlan MMD² becslés a referencia és az új adatok között."""
    if sigma is None:
        sigma = _median_sigma(torch.cat([X_ref, X_new]))
    n, m = len(X_ref), len(X_new)
    Krr = _rbf_kernel(X_ref, X_ref, sigma)
    Knn = _rbf_kernel(X_new, X_new, sigma)
    Krn = _rbf_kernel(X_ref, X_new, sigma)
    # Torzítatlan becslés: az átlós elemeket kizárjuk (egy pont önmagával vett kernele torzít)
    return (
        (Krr.sum() - Krr.trace()) / (n * (n - 1))
      + (Knn.sum() - Knn.trace()) / (m * (m - 1))
      - 2 * Krn.mean()
    ).item(), sigma


def permutation_mmd_threshold(X_ref, X_new, n_perm=300, alpha=0.05):
    """Permutációs teszttel meghatározza, mekkora MMD értéknél kell riasztani."""
    combined = torch.cat([X_ref, X_new], dim=0)
    n = len(X_ref)
    sigma = _median_sigma(combined)
    obs_mmd, _ = mmd_statistic(X_ref, X_new, sigma)
    null_mmds = []
    for _ in range(n_perm):
        # Véletlenszerű permutáció → mit kapnánk ha nem lenne drift?
        perm = torch.randperm(len(combined))
        m, _ = mmd_statistic(combined[perm[:n]], combined[perm[n:]], sigma)
        null_mmds.append(m)
    threshold = float(np.quantile(null_mmds, 1 - alpha))  # 95. percentilis
    print(f"  MMD megfigyelt={obs_mmd:.6f}  null-95pct={threshold:.6f}  sigma={sigma:.3f}")
    return threshold, sigma


@torch.no_grad()
def get_embeddings(model, loader, device, max_batches=None):
    """Összegyűjti a modell belső reprezentációit az egész dataloaderen."""
    model.eval()
    embs = []
    for i, (X_batch, _) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        embs.append(model.get_embeddings(X_batch.to(device)).cpu())
    return torch.cat(embs)


# 3. KS-teszt (Kolmogorov Szmirnov)
# egyszerűbb statisztikai teszt: 
# összehasonlítja a modell konfidenciáját (a softmax maximum értékét) a referencia-eloszlással.

def ks_confidence_drift(ref_confs, new_confs, alpha=0.05):
    """Kolmogorov-Smirnov teszt: szignifikánsan eltér-e az önbizalom eloszlása?"""
    stat, p = ks_2samp(ref_confs, new_confs)
    return stat, p, bool(p < alpha)  # True = drift detektálva


# 4. CUSUM detektor: Kumulatív Összeg
# Ez egy folyamat-felügyeleti módszer.
# Nem egyetlen tesztet csinál, hanem folyamatosan figyeli a trendet.


class CUSUMDetector:
    """
    Page-féle CUSUM kontrolldiagram.
    Pozitív eltolódást keres (pl. ha a predikciós halmazok mérete nő = romlik a lefedettség).

    Paraméterek:
        mu0   – referencia átlag (mit várunk normál esetben)
        sigma – szórás a referencia adatokon
        delta – mekkora eltolódást akarunk detektálni (szórásnyi egységekben)
        h_mult – küszöbszorzó (nagyobb → ritkább, de megbízhatóbb riasztás)
    """
    def __init__(self, mu0: float, sigma: float, delta: float = 1.0, h_mult: float = 5.0):
        self.k   = delta * sigma / 2   # tolerancia: ennyi eltérést még elnézünk
        self.h   = h_mult * sigma      # döntési küszöb: ennél nagyobb S → riasztás
        self.mu0 = mu0
        self.S   = 0.0                 # kumulatív összeg (ez gyűlik fel drift esetén)
        self._step = 0
        self.alerts = []

    def update(self, value: float) -> bool:
        """Egy új értékkel frissíti a CUSUM statisztikát. True = riasztás."""
        # Ha az érték a vártnál (+ tolerancia) nagyobb, S nő; különben S nem csökken 0 alá
        self.S = max(0.0, self.S + (value - self.mu0) - self.k)
        self._step += 1
        if self.S > self.h:
            self.alerts.append(self._step)
            self.S = 0.0  # reset riasztás után – újra figyelünk
            return True
        return False

## 5. Tanítás + Konformal Kalibráció


Ha zajosak a képek, a modell bizonytalanabb lesz → a halmazok nagyobbak lesznek → ezt tudnánk detektálni!

In [6]:
ALPHA = 0.1  # 90%-os lefedettséget szeretnénk
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Eszköz: {device}")

# A tiszta trainset-et 80/20 arányban osztjuk tanítás és kalibráció között
n_clean    = len(clean_trainset)
train_size = int(0.8 * n_clean)
cal_size   = n_clean - train_size  # a maradék a kalibrációs halmaz

train_ds, cal_ds = random_split(clean_trainset, [train_size, cal_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
cal_loader   = DataLoader(cal_ds,   batch_size=64, shuffle=False)

# Modell és optimalizáló inicializálása
model     = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ── 1. fázis: Tanítás tiszta adatokon ────────────────────────────────────────
print("\n--- 1. FÁZIS: Tanítás tiszta adatokon ---")
for epoch in range(5):  # 5 epoch elég a bemutatóhoz; több epochkal jobb accuracy érhető el
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, cal_loader, criterion)
    print(f"Epoch {epoch+1:2d} | Train: loss={train_loss:.4f}, acc={train_acc:.3f} | "
          f"Kalibr: loss={val_loss:.4f}, acc={val_acc:.3f}")

# ── 2. fázis: Konformal kalibráció ───────────────────────────────────────────
print("\n--- 2. FÁZIS: Konformal küszöb meghatározása ---")
model.eval()
cal_scores = []
with torch.no_grad():
    for X_batch, y_batch in cal_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        # Non-conformity score: 1 - (az igazi osztály valószínűsége)
        # Ha a modell biztos és helyes → ez kicsi; ha bizonytalan → nagy
        true_class_probs = probs[torch.arange(len(y_batch)), y_batch]
        cal_scores.append(1.0 - true_class_probs)

cal_scores = torch.cat(cal_scores)

# Konformal kvantilis kiszámítása – ez lesz a döntési küszöb
n_cal   = len(cal_scores)
q_level = math.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal  # korrigált kvantilis szint
q_hat   = torch.quantile(cal_scores, min(q_level, 1.0))
print(f"Konformal küszöb q̂ = {q_hat:.4f}"
      f" → Minden 1 - p_c ≤ q̂ osztályt beleveszünk a predikciós halmazba")

Eszköz: cpu

--- 1. FÁZIS: Tanítás tiszta adatokon ---
Epoch  1 | Train: loss=1.6474, acc=0.397 | Kalibr: loss=1.3394, acc=0.519
Epoch  2 | Train: loss=1.3072, acc=0.530 | Kalibr: loss=1.1982, acc=0.578
Epoch  3 | Train: loss=1.1696, acc=0.584 | Kalibr: loss=1.0800, acc=0.620
Epoch  4 | Train: loss=1.0622, acc=0.623 | Kalibr: loss=1.0259, acc=0.641
Epoch  5 | Train: loss=0.9827, acc=0.652 | Kalibr: loss=1.0435, acc=0.637

--- 2. FÁZIS: Konformal küszöb meghatározása ---
Konformal küszöb q̂ = 0.9243 → Minden 1 - p_c ≤ q̂ osztályt beleveszünk a predikciós halmazba


## 6. Drift Detektorok Kalibrálása

Minden detektort be kell kalibrálni a tiszta adatokon

In [ ]:
print("--- Drift detektorok kalibrálása tiszta adatokon ---")

# (a) Referencia embeddingek kinyerése, majd PCA-val 32 dimenzióra csökkentve
ref_emb_raw = get_embeddings(model, cal_loader, device)   # (N, 256)
pca         = OnlinePCA(n_components=32).fit(ref_emb_raw) # PCA illesztése a ref. adatokra
ref_emb     = pca.transform(ref_emb_raw)                  # (N, 32)
print(f"Referencia embeddingek PCA után: {ref_emb.shape}")

# (b) Referencia önbizalom-eloszlás (a KS-teszthez)
model.eval()
ref_confs = []
with torch.no_grad():
    for X_batch, _ in cal_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        ref_confs.extend(probs.max(dim=1).values.tolist())  # max softmax = önbizalom
ref_confs = np.array(ref_confs)
print(f"Referencia önbizalom – átlag: {ref_confs.mean():.3f}  szórás: {ref_confs.std():.3f}")

# (c) MMD detekciós küszöb permutációs teszttel
# A referencia adatokat kettéfelezzük, és megnézzük, mekkora MMD-t kapunk "semmi-sem-változott" esetben
half = len(ref_emb) // 2
mmd_threshold, mmd_sigma = permutation_mmd_threshold(
    ref_emb[:half], ref_emb[half:], n_perm=300, alpha=0.05
)
print(f"MMD detekciós küszöb: {mmd_threshold:.6f}")

# (d) CUSUM alapvonal: a predikciós halmaz méretének átlaga és szórása tiszta adatokon
model.eval()
ref_set_sizes = []
with torch.no_grad():
    for X_batch, _ in cal_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        # Hány osztályt vesz be a konformal halmaz? 
        sizes = (probs >= (1.0 - q_hat)).sum(dim=1).float().tolist()
        ref_set_sizes.extend(sizes)

mu0   = float(np.mean(ref_set_sizes))
sigma = float(np.std(ref_set_sizes)) + 1e-6  # kis epsilon a nullaosztás ellen
cusum = CUSUMDetector(mu0=mu0, sigma=sigma, delta=1.0, h_mult=5.0)
print(f"CUSUM alapvonal – mu0: {mu0:.3f}  sigma: {sigma:.3f}  k: {cusum.k:.3f}  h: {cusum.h:.3f}")

--- Drift detektorok kalibrálása tiszta adatokon ---
Referencia embeddingek PCA után: torch.Size([10000, 32])
Referencia önbizalom – átlag: 0.651  szórás: 0.224
MMD null-eloszlás kiszámítása (~10 másodperc)...
  MMD megfigyelt=0.000106  null-95pct=0.000110  sigma=12.703
MMD detekciós küszöb: 0.000110  sigma: 12.703
CUSUM alapvonal – mu0: 2.484  sigma: 1.224  k: 0.612  h: 6.119


## 7. Igazi proba

Először 20 batch tiszta adat jön, majd 20 batch zajos adat.

A detektorok nem tudják mikor vált az adatforrás, ezt kell felismerni

aamit nézünk
- `SetSz` – átlagos predikciós halmaz méret (ha nő → a modell bizonytalanabb lett)
- `CUSUM-S` – a felhalmozott eltérés (ha meghaladja a küszöböt → riasztás)
- `MMD` – feature-tér távolság (5 batchenként számítjuk, mert mintatszükséges)
- `KS-p` – p-érték az önbizalom-eloszlásra (ha p < 0.05 → szignifikáns változás)

In [ ]:
clean_test_loader = DataLoader(clean_testset, batch_size=64, shuffle=False)
noisy_test_loader = DataLoader(noisy_dataset, batch_size=64, shuffle=False)

# Adatfolyam összerakása: 20 clean batch, majd 20 noisy batch
stream_batches = []
for i, batch in enumerate(clean_test_loader):
    if i >= 20: break
    stream_batches.append(("CLEAN", batch))
for i, batch in enumerate(noisy_test_loader):
    if i >= 20: break
    stream_batches.append(("NOISY", batch))

# Az MMD és KS több mintát igényel → ablakos megközelítés
WINDOW = 5  # ennyi batch adatát gyűjtjük össze mielőtt tesztelünk

window_emb_list  = []  # az aktuális ablak embeddingei
window_conf_list = []  # az összes eddig látott önbizalom-érték (KS-hez)
log = []

model.eval()
print(f"{'Lépés':>5}  {'Forrás':>5}  {'SetSz':>5}  {'CUSUM-S':>7}  {'MMD':>8}  {'KS-p':>6}  Riasztás")
print("-" * 72)

with torch.no_grad():
    for step, (data_type, (X_batch, y_batch)) in enumerate(stream_batches):
        X_batch = X_batch.to(device)
        probs   = torch.softmax(model(X_batch), dim=1).cpu()
        emb_raw = model.get_embeddings(X_batch).cpu()
        emb     = pca.transform(emb_raw)  # vetítés a referencia PCA-térbe

        # Konformal halmaz mérete
        pred_sets = probs >= (1.0 - q_hat)
        avg_size  = pred_sets.sum(dim=1).float().mean().item()

        # Ablak frissítése
        window_emb_list.append(emb)
        window_conf_list.extend(probs.max(dim=1).values.tolist())

        alerts  = []
        mmd_val = float('nan')
        ks_p    = float('nan')

        # CUSUM: minden batchnél frissítjük
        if cusum.update(avg_size):
            alerts.append("CUSUM")

        # MMD + KS: csak ha elegendő minta gyűlt össze
        if len(window_emb_list) >= WINDOW:
            new_emb  = torch.cat(window_emb_list[-WINDOW:])
            new_conf = np.array(window_conf_list)

            # Véletlenszerű alminta a ref. embeddingekből (azonos méret kell)
            idx = torch.randperm(len(ref_emb))[:len(new_emb)]
            mmd_val, _ = mmd_statistic(ref_emb[idx], new_emb, sigma=mmd_sigma)
            if mmd_val > mmd_threshold:
                alerts.append("MMD")

            # KS teszt: az összes eddig látott önbizalom vs. referencia
            _, ks_p, ks_drift = ks_confidence_drift(ref_confs, new_conf)
            if ks_drift:
                alerts.append("KS")

        alert_str = ", ".join(alerts) if alerts else "—"
        mmd_str   = f"{mmd_val:8.5f}" if not np.isnan(mmd_val) else "     nan"
        ks_str    = f"{ks_p:6.3f}"    if not np.isnan(ks_p)    else "   nan"
        print(f"{step:>5}  {data_type:>5}  {avg_size:>5.2f}  {cusum.S:>7.3f}  {mmd_str}  {ks_str}  {alert_str}")

        log.append(dict(step=step, data_type=data_type, avg_size=avg_size,
                        cusum_S=cusum.S, mmd=mmd_val, ks_p=ks_p, alert=bool(alerts)))

Lépés  Forrás  SetSz  CUSUM-S       MMD    KS-p  Riasztás
------------------------------------------------------------------------
    0  CLEAN   2.36    0.000       nan     nan  —
    1  CLEAN   2.56    0.000       nan     nan  —
    2  CLEAN   2.45    0.000       nan     nan  —
    3  CLEAN   2.72    0.000       nan     nan  —
    4  CLEAN   2.33    0.000   0.00068   0.928  MMD
    5  CLEAN   2.44    0.000  -0.00105   0.779  —
    6  CLEAN   2.41    0.000   0.00046   0.852  MMD
    7  CLEAN   2.64    0.000  -0.00046   0.715  —
    8  CLEAN   2.36    0.000   0.00016   0.692  MMD
    9  CLEAN   2.44    0.000   0.00012   0.506  MMD
   10  CLEAN   2.61    0.000  -0.00065   0.517  —
   11  CLEAN   2.41    0.000   0.00032   0.755  MMD
   12  CLEAN   2.56    0.000   0.00106   0.696  MMD
   13  CLEAN   2.45    0.000  -0.00116   0.796  —
   14  CLEAN   2.59    0.000  -0.00091   0.876  —
   15  CLEAN   2.39    0.000  -0.00012   0.744  —
   16  CLEAN   2.31    0.000  -0.00012   0.634  —
   17  

In [10]:
steps      = [r['step']     for r in log]
set_sizes  = [r['avg_size'] for r in log]
cusum_vals = [r['cusum_S']  for r in log]
mmd_vals   = [r['mmd'] if not (r['mmd'] != r['mmd']) else 0 for r in log]  # nan → 0
ks_ps      = [r['ks_p'] if not (r['ks_p'] != r['ks_p']) else 1 for r in log]
alerts     = [r['step'] for r in log if r['alert']]

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
transition = 20  # ezen a lépésen vált a tiszta → zajos adatra

# Háttérszínezés és riasztásvonalak minden panelre
for ax in axes:
    ax.axvspan(transition, max(steps) + 1, color='salmon', alpha=0.15, label='Zajos adatok')
    for a in alerts:
        ax.axvline(a, color='red', linewidth=0.8, alpha=0.6)

# 1. panel: Konformal halmaz mérete + CUSUM
axes[0].plot(steps, set_sizes,  label='Átl. halmaz méret', color='steelblue')
axes[0].plot(steps, cusum_vals, label='CUSUM S',            color='darkorange', linestyle='--')
axes[0].axhline(mu0, color='green', linestyle=':', label=f'Alapvonal μ₀={mu0:.2f}')
axes[0].set_ylabel('Halmaz méret / CUSUM')
axes[0].set_title('Konformal Halmaz Mérete és CUSUM Detektor')
axes[0].legend(fontsize=8)

# 2. panel: MMD értékek
axes[1].plot(steps, mmd_vals, label='MMD²', color='purple')
axes[1].axhline(mmd_threshold, color='red', linestyle='--',
                label=f'Küszöb ({mmd_threshold:.5f})')
axes[1].set_ylabel('MMD²')
axes[1].set_title('MMD Kovariát-eltolódás Detektor (belső reprezentációkon)')
axes[1].legend(fontsize=8)

# 3. panel: KS p-értékek
axes[2].plot(steps, ks_ps, label='KS p-érték', color='teal')
axes[2].axhline(0.05, color='red', linestyle='--', label='α = 0.05 (szignifikancia)')
axes[2].set_ylabel('p-érték')
axes[2].set_xlabel('Batch lépés')
axes[2].set_title('KS-teszt az Önbizalom-eloszláson')
axes[2].legend(fontsize=8)

plt.suptitle('Drift Detekció: Tiszta → Zajos Adatok (CIFAR-10-C)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('drift_timeline.png', dpi=130, bbox_inches='tight')
plt.show()
print("Mentve: drift_timeline.png")
print(f"\nÖsszes riasztás: {len(alerts)} db, lépéseknél: {alerts}")

Mentve: drift_timeline.png

Összes riasztás: 26 db, lépéseknél: [4, 6, 8, 9, 11, 12, 18, 19, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]


C:\Users\anna.gavrilova.TENZORTECH\AppData\Local\Temp\ipykernel_15644\2494686981.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
